# 04 - Predicciones del mejor modelo semanal

**Modelo ganador:** Ridge con `alpha=5000` sobre `df_semanal.csv` (sin lags).  
**Split:** Train hasta `2024-W29`, predicciones walk-forward desde `2024-W30`.  
**Objetivo:** Guardar en Google Sheets una fila por prediccion individual (paso de walk-forward) para Ridge, Baseline y Zeros.

In [ ]:
# pip install gspread google-auth pandas scikit-learn

In [ ]:
import os
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import gspread

from google.oauth2.service_account import Credentials
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler

import warnings
warnings.filterwarnings('ignore', category=UserWarning)

In [ ]:
# CONFIGURACION
DATA_DIR             = '../../Datos_csv'
SERVICE_ACCOUNT_FILE = 'credenciales_google.json'

SHEET_URL = 'https://docs.google.com/spreadsheets/d/1NTToszrpWdyN0oXEORs4_7o1Uk_j7eG-5Rc4VD3gulE/edit?gid=0#gid=0'

SCOPES = [
    'https://www.googleapis.com/auth/spreadsheets',
    'https://www.googleapis.com/auth/drive',
]

# Modelo ganador
ARCHIVO = 'df_semanal.csv'
METODO  = 'Ridge'
ALPHA   = 5000

# Split temporal (igual que en 02_comparacion)
FECHA_TRAIN_FIN = '2024-W29'
FECHA_TEST_INI  = '2024-W30'

# ETFs excluidos: el modelo Ridge no mejora al predictor nulo (Zeros) en RMSE.
# Un RMSE >= RMSE_zeros indica que las predicciones no contienen señal util
# y distorsionarian la optimizacion de Markowitz al penalizar errores grandes.
ETFS_EXCLUIDOS = {
    'target_USO', 'target_VNQ', 'target_TLT', 'target_XLU',
    'target_XLRE', 'target_XLE', 'target_EEM', 'target_FXI',
    'target_MCHI', 'target_AGG', 'target_IEF',
}

# Cabecera del Google Sheet (15 columnas)
HEADER = [
    'Archivo', 'Metodo', 'Target',
    'RMSE', 'RMSE_baseline', 'RMSE_zeros',
    'MAE',  'MAE_baseline',  'MAE_zeros',
    'Alpha',
    'Valor_Real', 'Prediccion', 'Valor_Baseline', 'Valor_Zeros', 'Fecha_Semana'
]

In [ ]:
# GOOGLE SHEETS
def conectar_sheet():
    creds = Credentials.from_service_account_file(SERVICE_ACCOUNT_FILE, scopes=SCOPES)
    client = gspread.authorize(creds)
    return client.open_by_url(SHEET_URL).get_worksheet(0)


def leer_predicciones_existentes():
    ws = conectar_sheet()
    records = ws.get_all_records()
    if not records:
        return pd.DataFrame(columns=HEADER)
    return pd.DataFrame(records)


def guardar_predicciones(df_existing, new_rows):
    if not new_rows:
        print('  Sin filas nuevas que guardar.')
        return
    ws = conectar_sheet()
    if df_existing.empty:
        ws.clear()
        ws.update(range_name='A1', values=[HEADER] + new_rows)
        print(f'  Primera escritura: {len(new_rows)} filas.')
    else:
        ws.append_rows(new_rows, value_input_option='RAW')
        print(f'  Añadidas {len(new_rows)} filas nuevas al sheet.')

In [ ]:
# PREPROCESADO
def preprocess_weekly_data(df, target_actual):
    df = df.copy()
    if target_actual not in df.columns:
        raise ValueError(f'Target {target_actual} no encontrado.')

    week_key = df['week_key'].copy()
    y = df[target_actual].copy()

    df = df.drop(columns=['week_key', 'year', 'semana', 'mes'], errors='ignore')
    target_cols = [c for c in df.columns if c.startswith('target_')]
    X = df.drop(columns=target_cols, errors='ignore')
    X = X.select_dtypes(include=[np.number])

    mask = pd.concat([X, y.rename(target_actual)], axis=1).notna().all(axis=1)
    X        = X.loc[mask].reset_index(drop=True)
    y        = y.loc[mask].reset_index(drop=True)
    week_key = week_key.loc[mask].reset_index(drop=True)

    return X, y, week_key


def temporal_split_por_weekkey(X, y, week_key, train_fin, test_ini):
    week_key = week_key.astype(str)
    mask_train = week_key <= train_fin
    mask_test  = week_key >= test_ini

    X_train  = X.loc[mask_train].copy().reset_index(drop=True)
    X_test   = X.loc[mask_test].copy().reset_index(drop=True)
    y_train  = y.loc[mask_train].copy().reset_index(drop=True)
    y_test   = y.loc[mask_test].copy().reset_index(drop=True)
    wk_train = week_key.loc[mask_train].copy().reset_index(drop=True)
    wk_test  = week_key.loc[mask_test].copy().reset_index(drop=True)

    if len(X_train) == 0 or len(X_test) == 0:
        raise ValueError(f'Split vacio con train_fin={train_fin}, test_ini={test_ini}')

    return X_train, X_test, y_train, y_test, wk_train, wk_test

In [ ]:
# FORMATO FECHA SEMANA
def formatear_week_key(week_key: str) -> str:
    if not week_key or not str(week_key).strip():
        return ''
    wk     = str(week_key).strip()
    inicio = datetime.strptime(wk + '-1', '%G-W%V-%u')
    fin    = inicio + timedelta(days=6)
    return f'{wk} ({inicio.strftime("%Y-%m-%d")}/{fin.strftime("%Y-%m-%d")})'

In [ ]:
# WALK-FORWARD CON METRICAS INDIVIDUALES POR PASO
def walk_forward_predicciones(X_train, y_train, X_test, y_test, wk_test, alpha):
    X_hist  = X_train.copy().reset_index(drop=True)
    y_hist  = y_train.copy().reset_index(drop=True)
    X_test  = X_test.copy().reset_index(drop=True)
    y_test  = y_test.copy().reset_index(drop=True)
    wk_test = wk_test.copy().reset_index(drop=True)

    pasos = []

    for i in range(len(X_test)):
        scaler    = StandardScaler()
        X_hist_sc = scaler.fit_transform(X_hist)
        x_next_sc = scaler.transform(X_test.iloc[[i]])

        model = Ridge(alpha=alpha)
        model.fit(X_hist_sc, y_hist)

        pred_model    = float(model.predict(x_next_sc)[0])
        pred_baseline = float(y_hist.iloc[-1])
        pred_zeros    = 0.0
        real          = float(y_test.iloc[i])

        # Metricas individuales para este unico paso
        err_m = real - pred_model
        err_b = real - pred_baseline
        err_z = real - pred_zeros

        pasos.append({
            'week_key':      str(wk_test.iloc[i]),
            'real':          real,
            'pred_model':    pred_model,
            'pred_baseline': pred_baseline,
            'pred_zeros':    pred_zeros,
            'rmse_m': float(np.sqrt(err_m ** 2)),
            'rmse_b': float(np.sqrt(err_b ** 2)),
            'rmse_z': float(np.sqrt(err_z ** 2)),
            'mae_m':  float(abs(err_m)),
            'mae_b':  float(abs(err_b)),
            'mae_z':  float(abs(err_z)),
        })

        X_hist = pd.concat([X_hist, X_test.iloc[[i]]], ignore_index=True)
        y_hist = pd.concat([y_hist, pd.Series([real])],  ignore_index=True)

    return pasos

## ETFs excluidos del universo de prediccion

Se excluyeron los ETFs para los que el modelo Ridge **no mejora al predictor nulo** (prediccion constante igual a cero). Como los retornos financieros oscilan alrededor de cero, superar ese benchmark minimo es condicion necesaria para considerar que las predicciones contienen señal util.

El criterio de exclusion es **RMSE_modelo >= RMSE_zeros**: si el modelo comete errores mayores que simplemente predecir cero, sus predicciones introducen ruido en la optimizacion de Markowitz. El RMSE penaliza especialmente los errores grandes, y en una cartera un error grande en el retorno esperado puede distorsionar el peso asignado a ese activo.

**ETFs excluidos:** AGG, IEF, TLT, VNQ, USO, XLU, XLRE, XLE, EEM, FXI, MCHI

In [ ]:
# EJECUCION PRINCIPAL
print('Leyendo predicciones existentes del sheet...')
df_existing = leer_predicciones_existentes()
print(f'  Filas ya en sheet: {len(df_existing)}')
print()

print(f'Cargando {ARCHIVO}...')
df = pd.read_csv(os.path.join(DATA_DIR, ARCHIVO))

targets = sorted([c for c in df.columns if c.startswith('target_')])
targets = [t for t in targets if t not in ETFS_EXCLUIDOS]
print(f'Targets activos     : {len(targets)}')
print(f'Modelo              : {METODO} | Alpha: {ALPHA}')
print(f'Split               : train <= {FECHA_TRAIN_FIN}  |  test >= {FECHA_TEST_INI}')
print('-' * 65)

all_new_rows = []
n_test_semanas = None

for idx, target in enumerate(targets, 1):
    ticker = target.replace('target_', '')

    X, y, week_key = preprocess_weekly_data(df, target)
    X_train, X_test, y_train, y_test, wk_train, wk_test = temporal_split_por_weekkey(
        X, y, week_key, FECHA_TRAIN_FIN, FECHA_TEST_INI
    )

    # Semanas del test disponibles en el CSV
    weeks_disponibles = {formatear_week_key(w) for w in wk_test.tolist()}

    # Semanas ya guardadas en el sheet para este target
    if not df_existing.empty and 'Target' in df_existing.columns:
        weeks_guardadas = set(
            df_existing[
                (df_existing['Target'] == target) &
                (df_existing['Metodo'] == METODO)
            ]['Fecha_Semana'].astype(str).tolist()
        )
    else:
        weeks_guardadas = set()

    weeks_nuevas = weeks_disponibles - weeks_guardadas

    if n_test_semanas is None:
        n_test_semanas = len(weeks_disponibles)

    if not weeks_nuevas:
        print(f'[{idx:2d}/{len(targets)}] {ticker:6s} | ya predicho ({len(weeks_guardadas)} semanas). Sin cambios.')
        continue

    # Walk-forward completo (necesario para el estado correcto del modelo en semanas nuevas)
    pasos = walk_forward_predicciones(
        X_train, y_train, X_test, y_test, wk_test, alpha=ALPHA
    )

    filas_añadidas = 0
    for paso in pasos:
        fecha_str = formatear_week_key(paso['week_key'])
        if fecha_str not in weeks_nuevas:
            continue
        all_new_rows.append([
            ARCHIVO, METODO, target,
            round(paso['rmse_m'], 6), round(paso['rmse_b'], 6), round(paso['rmse_z'], 6),
            round(paso['mae_m'],  6), round(paso['mae_b'],  6), round(paso['mae_z'],  6),
            ALPHA,
            round(paso['real'],          6),
            round(paso['pred_model'],    6),
            round(paso['pred_baseline'], 6),
            0.0,
            fecha_str
        ])
        filas_añadidas += 1

    print(
        f'[{idx:2d}/{len(targets)}] {ticker:6s} | '
        f'semanas nuevas={len(weeks_nuevas)} | '
        f'ya guardadas={len(weeks_guardadas)}'
    )

# DataFrame completo para el grafico (existente + nuevas)
if all_new_rows:
    df_new = pd.DataFrame(all_new_rows, columns=HEADER)
    df_all_preds = pd.concat([df_existing, df_new], ignore_index=True) if not df_existing.empty else df_new
else:
    df_all_preds = df_existing.copy()

print('-' * 65)
print(f'Filas nuevas generadas : {len(all_new_rows)}')
print(f'Total filas para plot  : {len(df_all_preds)}')

In [ ]:
# GUARDAR EN GOOGLE SHEETS (solo filas nuevas)
guardar_predicciones(df_existing, all_new_rows)
print('Sheet actualizado:', SHEET_URL)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# Preparar datos: retornos semanales en % (x100 para legibilidad)
df_p = df_all_preds[df_all_preds['Metodo'] == METODO].copy()

df_p['fecha'] = pd.to_datetime(
    df_p['Fecha_Semana'].str.extract(r'\((\d{4}-\d{2}-\d{2})/')[0]
)
df_p['Valor_Real'] = pd.to_numeric(df_p['Valor_Real'], errors='coerce') * 100
df_p['Prediccion'] = pd.to_numeric(df_p['Prediccion'], errors='coerce') * 100

tickers_plot = sorted(df_p['Target'].str.replace('target_', '', regex=False).unique())
n = len(tickers_plot)

fig, axes = plt.subplots(n, 1, figsize=(14, n * 2.8))
if n == 1:
    axes = [axes]

for i, ticker in enumerate(tickers_plot):
    ax = axes[i]
    sub = df_p[df_p['Target'] == f'target_{ticker}'].sort_values('fecha')
    ax.plot(sub['fecha'], sub['Valor_Real'], label='Real',       color='steelblue', linewidth=1.2)
    ax.plot(sub['fecha'], sub['Prediccion'], label='Ridge pred', color='tomato',    linewidth=1.0, linestyle='--')
    ax.axhline(0, color='gray', linewidth=0.5, linestyle=':')
    ax.set_title(ticker, fontsize=9, fontweight='bold', loc='left')
    ax.set_ylabel('Retorno\nsemanal (%)', fontsize=7)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.1f}%'))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
    ax.tick_params(axis='x', labelsize=7, rotation=45)
    ax.tick_params(axis='y', labelsize=7)
    ax.legend(fontsize=7, loc='upper right')

fig.suptitle(
    f'Retorno semanal real vs prediccion Ridge  (alpha={ALPHA} | test desde {FECHA_TEST_INI})',
    fontsize=12, fontweight='bold'
)
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import cohen_kappa_score, accuracy_score
import matplotlib.pyplot as plt

df_ridge = df_all_preds[df_all_preds['Metodo'] == METODO].copy()
df_ridge['Valor_Real']     = pd.to_numeric(df_ridge['Valor_Real'],     errors='coerce')
df_ridge['Prediccion']     = pd.to_numeric(df_ridge['Prediccion'],     errors='coerce')
df_ridge['Valor_Baseline'] = pd.to_numeric(df_ridge['Valor_Baseline'], errors='coerce')

metricas_dir = []

for target, grupo in df_ridge.groupby('Target'):
    ticker = target.replace('target_', '')
    grupo = grupo.dropna(subset=['Valor_Real', 'Prediccion', 'Valor_Baseline'])
    if len(grupo) < 2:
        continue

    real_dir     = (grupo['Valor_Real']     > 0).astype(int)
    ridge_dir    = (grupo['Prediccion']     > 0).astype(int)
    baseline_dir = (grupo['Valor_Baseline'] > 0).astype(int)

    metricas_dir.append({
        'ETF':               ticker,
        'Kappa_Ridge':       round(cohen_kappa_score(real_dir, ridge_dir),    4),
        'Kappa_Baseline':    round(cohen_kappa_score(real_dir, baseline_dir), 4),
        'Accuracy_Ridge':    f'{accuracy_score(real_dir, ridge_dir):.1%}',
        'Accuracy_Baseline': f'{accuracy_score(real_dir, baseline_dir):.1%}',
        'n_semanas':         len(grupo),
    })

df_dir = (
    pd.DataFrame(metricas_dir)
    .set_index('ETF')
    .sort_values('Kappa_Ridge', ascending=False)
)

print('Analisis Direccional Ridge (Sube/Baja):')
display(df_dir)

# Gráfico: Kappa_Ridge vs Real — una barra por ETF, verde si positivo, rojo si negativo
fig, ax = plt.subplots(figsize=(14, 5))
x = range(len(df_dir))
colores = ['#4CAF50' if v >= 0 else '#F44336' for v in df_dir['Kappa_Ridge']]
ax.bar(x, df_dir['Kappa_Ridge'], color=colores, alpha=0.85)
ax.axhline(0,    color='black', linewidth=0.9)
ax.axhline(0.20, color='gray',  linewidth=0.7, linestyle='--', label='Acuerdo leve (0.20)')
ax.axhline(0.40, color='gray',  linewidth=0.7, linestyle=':',  label='Acuerdo moderado (0.40)')
ax.set_xticks(list(x))
ax.set_xticklabels(df_dir.index, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Kappa de Cohen')
ax.set_title('Kappa de Cohen — Ridge vs Real (dirección sube/baja)', fontweight='bold')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## Interpretacion: Kappa de Cohen en prediccion direccional

### Que mide la Kappa de Cohen

La **Kappa de Cohen (κ)** mide el acuerdo entre la prediccion y la realidad descontando el acuerdo que se obtendria **solo por azar**. A diferencia del simple *accuracy*, la kappa penaliza que un modelo acierte por suerte cuando las clases estan desbalanceadas (mas semanas de subida que de bajada, o viceversa).

| Valor κ | Interpretacion |
|---------|----------------|
| < 0     | Peor que el azar |
| 0.00 – 0.20 | Acuerdo leve |
| 0.21 – 0.40 | Acuerdo moderado |
| 0.41 – 0.60 | Acuerdo sustancial |
| > 0.60  | Acuerdo casi perfecto |

En prediccion de retornos financieros, donde el mercado es eficiente, **cualquier κ positivo indica que el modelo captura algun patron direccional real**. Valores κ > 0.10 ya son relevantes en este contexto.

### Que vemos en nuestros resultados

- **ETFs con κ_Ridge > 0**: el modelo acierta la direccion semanal mejor que el azar. Son los candidatos mas solidos para Markowitz porque una prediccion de retorno esperado con sesgo direccional correcto mejora los pesos de la cartera.

- **ETFs con κ_Ridge < 0 o ≈ 0**: el modelo no distingue subidas de bajadas. Incluirlos en Markowitz podria asignar retornos esperados con sesgo incorrecto, lo que es mas danino que usar directamente cero.

- **Comparacion con Baseline**: si κ_Ridge > κ_Baseline, el modelo aporta informacion mas alla de la inercia del mercado (el ultimo retorno conocido). Si κ_Baseline > κ_Ridge en algun ETF, ese activo tiene mas inercia que señal lineal y bastaria con el predictor naive.

> **Nota:** El predictor Zeros no se incluye porque siempre predice retorno = 0 (sin direccion), lo que hace inviable el calculo de kappa.

In [ ]:
from scipy import stats
import matplotlib.pyplot as plt

df_ic = df_all_preds[df_all_preds['Metodo'] == METODO].copy()
df_ic['Valor_Real'] = pd.to_numeric(df_ic['Valor_Real'], errors='coerce')
df_ic['Prediccion'] = pd.to_numeric(df_ic['Prediccion'], errors='coerce')

registros_ic = []
for target, grupo in df_ic.groupby('Target'):
    ticker = target.replace('target_', '')
    grupo = grupo.dropna(subset=['Valor_Real', 'Prediccion'])
    if len(grupo) < 5:
        continue
    real = grupo['Valor_Real'].values
    pred = grupo['Prediccion'].values
    ic_p,  pv_p  = stats.pearsonr(pred, real)
    ic_s,  pv_s  = stats.spearmanr(pred, real)
    registros_ic.append({
        'ETF': ticker,
        'IC_Pearson':  round(ic_p, 4),
        'IC_Spearman': round(ic_s, 4),
        'p_Pearson':   round(pv_p, 4),
        'p_Spearman':  round(pv_s, 4),
        'n': len(grupo),
    })

df_ic_res = pd.DataFrame(registros_ic).set_index('ETF').sort_values('IC_Pearson', ascending=False)

# IC cross-sectional: por semana, correlacion Spearman entre pred y real de todos los ETFs
df_cross = df_ic.copy()
df_cross['week_key'] = df_cross['Fecha_Semana'].str.extract(r'^(\d{4}-W\d+)')
ic_cross_semana = []
for semana, g in df_cross.groupby('week_key'):
    g = g.dropna(subset=['Valor_Real', 'Prediccion'])
    if len(g) < 5:
        continue
    r, _ = stats.spearmanr(g['Prediccion'], g['Valor_Real'])
    ic_cross_semana.append({'semana': semana, 'IC_cross': r})

df_cross_ic   = pd.DataFrame(ic_cross_semana)
ic_cross_mean = df_cross_ic['IC_cross'].mean()
ic_cross_std  = df_cross_ic['IC_cross'].std()
ir_ratio      = ic_cross_mean / ic_cross_std if ic_cross_std > 0 else 0

n_pos_p = (df_ic_res['IC_Pearson']  > 0).sum()
n_pos_s = (df_ic_res['IC_Spearman'] > 0).sum()
n_sig   = (df_ic_res['p_Pearson']   < 0.10).sum()
n_total = len(df_ic_res)

print('=' * 60)
print('INFORMATION COEFFICIENT (IC) — Ridge semanal')
print('=' * 60)
print(f'  ETFs con IC Pearson  > 0 : {n_pos_p}/{n_total}')
print(f'  ETFs con IC Spearman > 0 : {n_pos_s}/{n_total}')
print(f'  ETFs con p-valor < 0.10  : {n_sig}/{n_total}')
print()
print(f'  IC cross-sectional medio : {ic_cross_mean:.4f}')
print(f'  Std IC cross-sectional   : {ic_cross_std:.4f}')
print(f'  Information Ratio (IC/std): {ir_ratio:.4f}')
print()
print('  Referencia: IC>0.05 util | IC>0.10 bueno | IR>0 valor en promedio')
print('=' * 60)
display(df_ic_res)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

ax = axes[0]
x = range(len(df_ic_res))
ax.bar([i - 0.2 for i in x], df_ic_res['IC_Pearson'],  width=0.4,
       color=['tomato' if v > 0 else '#aaa' for v in df_ic_res['IC_Pearson']],
       alpha=0.85, label='Pearson')
ax.bar([i + 0.2 for i in x], df_ic_res['IC_Spearman'], width=0.4,
       color=['steelblue' if v > 0 else '#ccc' for v in df_ic_res['IC_Spearman']],
       alpha=0.7, label='Spearman')
ax.axhline(0,    color='black',     linewidth=0.8)
ax.axhline(0.05, color='green',     linewidth=0.8, linestyle='--', label='IC=0.05 (util)')
ax.axhline(0.10, color='darkgreen', linewidth=0.8, linestyle=':',  label='IC=0.10 (bueno)')
ax.set_xticks(list(x))
ax.set_xticklabels(df_ic_res.index, rotation=90, fontsize=7)
ax.set_title('IC por ETF (Pearson y Spearman)', fontsize=10, fontweight='bold')
ax.set_ylabel('Information Coefficient')
ax.legend(fontsize=7)

ax2 = axes[1]
ax2.hist(df_cross_ic['IC_cross'], bins=25, color='steelblue', alpha=0.8, edgecolor='white')
ax2.axvline(0,             color='red',      linewidth=1.2, linestyle='--', label='IC=0')
ax2.axvline(ic_cross_mean, color='darkblue', linewidth=1.5, label=f'Media={ic_cross_mean:.3f}')
ax2.set_title('IC cross-sectional por semana\n(Spearman entre ETFs)', fontsize=10, fontweight='bold')
ax2.set_xlabel('IC Spearman')
ax2.set_ylabel('N semanas')
ax2.legend(fontsize=8)

ax3 = axes[2]
df_cs = df_cross_ic.sort_values('semana').reset_index(drop=True)
df_cs['IC_acum'] = df_cs['IC_cross'].cumsum()
ax3.plot(df_cs.index, df_cs['IC_acum'], color='steelblue', linewidth=1.5)
ax3.axhline(0, color='red', linewidth=0.8, linestyle='--')
ax3.fill_between(df_cs.index, df_cs['IC_acum'], 0,
                 where=df_cs['IC_acum'] >= 0, alpha=0.2, color='green')
ax3.fill_between(df_cs.index, df_cs['IC_acum'], 0,
                 where=df_cs['IC_acum'] < 0, alpha=0.2, color='red')
ax3.set_title('IC cross-sectional acumulado\n(tendencia + = senal persistente)', fontsize=10, fontweight='bold')
ax3.set_xlabel('Semana (orden cronologico)')
ax3.set_ylabel('IC acumulado')

plt.suptitle('Information Coefficient — señal para Black-Litterman', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## IC (Information Coefficient) y su uso en Black-Litterman

### Por que el IC importa mas que la Kappa para BL

La **Kappa de Cohen** mide si el modelo acierta la *direccion* (sube/baja). Pero **Black-Litterman no necesita que aciertes la direccion** — necesita que tus predicciones tengan *correlacion positiva* con los retornos reales, aunque sea pequeña.

| Metrica | Pregunta | Relevancia para BL |
|---------|----------|--------------------|
| Kappa de Cohen | ¿Aciertas sube/baja? | Baja |
| IC Pearson | ¿Hay correlacion lineal pred vs real? | **Alta** |
| IC Spearman cross-sectional | ¿El modelo ordena bien los ETFs cada semana? | **Muy alta** |
| Information Ratio (IC/std) | ¿La señal es consistente en el tiempo? | **Alta** |

### Umbrales de referencia en quant finance

- **IC > 0.05**: señal util, justifica usar el modelo como view en BL
- **IC > 0.10**: señal buena, confianza media-alta en las views
- **IR > 0**: el IC medio es positivo — las views añaden valor en promedio
- **IC acumulado con tendencia creciente**: la señal es persistente, no aleatoria

### Como se calibra Omega con el IC

En BL el parametro **Omega** (incertidumbre de cada view) se puede derivar del IC: mayor IC → Omega mas pequeño → mas peso a la prediccion del modelo. Un IC bajo pero positivo implica un Omega alto, lo que hace que el portfolio se aleje poco del equilibrio de mercado pero en la direccion correcta.

> Si el IC cross-sectional medio es positivo y el IC acumulado tiene tendencia creciente, el modelo Ridge **justifica su uso en Black-Litterman** incluso con Kappa baja. La señal existe aunque sea debil — y BL esta disenado exactamente para aprovechar señales debiles de forma robusta.

## Resumen global y por grupos de activos — Ridge 0 lags

Se agregan las métricas de error por ETF a partir de los pasos individuales del walk-forward y se comparan con los dos benchmarks de referencia (último valor y predicción nula). Los ETFs se clasifican en seis grupos para identificar qué categorías de activos resultan más o menos predecibles con este modelo.

In [ ]:
import numpy as np
import pandas as pd

# ── Clasificacion de ETFs por grupo de activos
GRUPOS_ETF = {
    'Core US':        ['SPY','IVV','VOO','QQQ','DIA','IWM','MDY','IJR','ITOT','VTI'],
    'Factores':       ['MTUM','QUAL','USMV','VLUE','IWF','IWD','VUG','VTV','VIG','DVY','SCHD'],
    'Sectores':       ['XLK','XLF','XLV','XLY','XLP','XLE','XLI','XLB','XLU','XLRE','VNQ'],
    'Internacional':  ['EFA','IEFA','VEA','EEM','IEMG','VWO','EWJ','EWG','EWQ','EWU','EWT','EWZ','FXI','MCHI','INDA'],
    'Renta fija':     ['AGG','BND','LQD','HYG','JNK','TLT','IEF','SHY','TIP'],
    'Materias primas':['GLD','IAU','SLV','USO','DBC'],
}
etf_to_grupo = {t: g for g, tks in GRUPOS_ETF.items() for t in tks}

# ── Cargar métricas por ETF ya calculadas
df_etf = pd.read_csv(os.path.join(DATA_DIR, 'resultados_ridge0lags.csv'))

df_etf['Grupo']            = df_etf['ETF'].map(etf_to_grupo).fillna('Otros')
df_etf['Mejora_base_pct']  = (df_etf['RMSE_BASE']  - df_etf['RMSE_Ridge']) / df_etf['RMSE_BASE']  * 100
df_etf['Mejora_zeros_pct'] = (df_etf['RMSE_ZEROS'] - df_etf['RMSE_Ridge']) / df_etf['RMSE_ZEROS'] * 100
df_etf['Bate_base']        = (df_etf['RMSE_Ridge'] < df_etf['RMSE_BASE']).astype(int)
df_etf['Bate_zeros']       = (df_etf['RMSE_Ridge'] < df_etf['RMSE_ZEROS']).astype(int)

# ═══════════════════════════════════════════
# TABLA GLOBAL
# ═══════════════════════════════════════════
tabla_global = pd.DataFrame({
    'RMSE': [df_etf['RMSE_Ridge'].mean(), df_etf['RMSE_BASE'].mean(),  df_etf['RMSE_ZEROS'].mean()],
    'MAE':  [df_etf['MAE_Ridge'].mean(),  df_etf['MAE_BASE'].mean(),   df_etf['MAE_ZEROS'].mean()],
}, index=['Ridge (0 lags)', 'Baseline (último valor)', 'Zeros (predicción nula)'])

print('━' * 55)
print('RESUMEN GLOBAL — Ridge 0 lags semanal')
print('━' * 55)
display(tabla_global.style.format('{:.6f}').highlight_min(color='lightgreen', axis=0))

n  = len(df_etf)
bb = df_etf['Bate_base'].sum()
bz = df_etf['Bate_zeros'].sum()
print(f'\n  ETFs totales          : {n}')
print(f'  Bate baseline         : {bb}/{n}  ({bb/n*100:.1f} %)')
print(f'  Bate zeros            : {bz}/{n}  ({bz/n*100:.1f} %)')
print(f'  Mejora media vs base  : {df_etf["Mejora_base_pct"].mean():.2f} %')
print(f'  Mejora media vs zeros : {df_etf["Mejora_zeros_pct"].mean():.2f} %')

# ═══════════════════════════════════════════
# TABLA POR GRUPO
# ═══════════════════════════════════════════
ORDEN_GRUPOS = ['Core US', 'Factores', 'Sectores', 'Internacional', 'Renta fija', 'Materias primas']

tabla_grupos = (
    df_etf.groupby('Grupo')
    .agg(
        N              =('ETF',             'count'),
        RMSE_Ridge     =('RMSE_Ridge',      'mean'),
        RMSE_BASE      =('RMSE_BASE',       'mean'),
        RMSE_ZEROS     =('RMSE_ZEROS',      'mean'),
        MAE_Ridge      =('MAE_Ridge',       'mean'),
        Mejora_base    =('Mejora_base_pct', 'mean'),
        Mejora_zeros   =('Mejora_zeros_pct','mean'),
        Bate_base      =('Bate_base',       'sum'),
        Bate_zeros     =('Bate_zeros',      'sum'),
    )
    .reindex([g for g in ORDEN_GRUPOS if g in df_etf['Grupo'].unique()])
    .reset_index()
)
tabla_grupos['Bate_base_str']  = tabla_grupos.apply(lambda r: f'{int(r.Bate_base)}/{int(r.N)}',  axis=1)
tabla_grupos['Bate_zeros_str'] = tabla_grupos.apply(lambda r: f'{int(r.Bate_zeros)}/{int(r.N)}', axis=1)

fmt = {c: '{:.5f}' for c in ['RMSE_Ridge','RMSE_BASE','RMSE_ZEROS','MAE_Ridge']}
fmt.update({'Mejora_base': '{:.2f} %', 'Mejora_zeros': '{:.2f} %'})

print('\n━' * 28)
print('POR GRUPO DE ACTIVOS')
print('━' * 55)
display(
    tabla_grupos[['Grupo','N','RMSE_Ridge','RMSE_BASE','RMSE_ZEROS',
                  'MAE_Ridge','Mejora_base','Mejora_zeros','Bate_base_str','Bate_zeros_str']]
    .rename(columns={
        'Bate_base_str':  'Bate base',
        'Bate_zeros_str': 'Bate zeros',
        'Mejora_base':    'Mejora vs base (%)',
        'Mejora_zeros':   'Mejora vs zeros (%)',
    })
    .style.format(fmt)
    .highlight_min(subset=['RMSE_Ridge'], color='lightgreen')
    .highlight_max(subset=['RMSE_Ridge'], color='#ffcccc')
)

# ═══════════════════════════════════════════
# RANKING POR ETF
# ═══════════════════════════════════════════
print('\n━' * 28)
print('RANKING POR ETF (ordenado por RMSE_Ridge)')
print('━' * 55)
display(
    df_etf[['Grupo','ETF','RMSE_Ridge','RMSE_BASE','RMSE_ZEROS',
            'Mejora_base_pct','Mejora_zeros_pct','Bate_base','Bate_zeros','n_semanas']]
    .sort_values('RMSE_Ridge')
    .reset_index(drop=True)
    .style.format({
        'RMSE_Ridge':       '{:.5f}',
        'RMSE_BASE':        '{:.5f}',
        'RMSE_ZEROS':       '{:.5f}',
        'Mejora_base_pct':  '{:.2f} %',
        'Mejora_zeros_pct': '{:.2f} %',
    })
    .highlight_min(subset=['RMSE_Ridge'], color='lightgreen')
    .highlight_max(subset=['RMSE_Ridge'], color='#ffcccc')
    .apply(lambda col: [
        'background-color: #d4edda' if v == 1 else 'background-color: #f8d7da'
        for v in col
    ], subset=['Bate_zeros'])
)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

COLORES_GRUPO = {
    'Core US':        '#2196F3',
    'Factores':       '#4CAF50',
    'Sectores':       '#FF9800',
    'Internacional':  '#9C27B0',
    'Renta fija':     '#00BCD4',
    'Materias primas':'#F44336',
}

df_plot = df_etf.sort_values(['Grupo', 'RMSE_Ridge']).reset_index(drop=True)
colores_barra = [COLORES_GRUPO.get(g, '#888') for g in df_plot['Grupo']]

fig, axes = plt.subplots(2, 1, figsize=(16, 11))

# ── Panel 1: RMSE por ETF (Ridge vs BASE vs ZEROS)
ax = axes[0]
x  = np.arange(len(df_plot))
w  = 0.28
ax.bar(x - w, df_plot['RMSE_Ridge'], width=w, label='Ridge',   color=colores_barra, alpha=0.90)
ax.bar(x,     df_plot['RMSE_BASE'],  width=w, label='Baseline',color='#90A4AE',     alpha=0.70)
ax.bar(x + w, df_plot['RMSE_ZEROS'], width=w, label='Zeros',   color='#CFD8DC',     alpha=0.70)
ax.set_xticks(x)
ax.set_xticklabels(df_plot['ETF'], rotation=45, ha='right', fontsize=7.5)
ax.set_ylabel('RMSE')
ax.set_title('RMSE por ETF — Ridge 0 lags vs benchmarks', fontweight='bold', fontsize=11)
ax.legend(fontsize=9)

# Separadores de grupo
grupo_anterior = None
for i, g in enumerate(df_plot['Grupo']):
    if g != grupo_anterior and i > 0:
        ax.axvline(i - 0.5, color='gray', linewidth=0.8, linestyle='--', alpha=0.6)
    grupo_anterior = g

leyenda_grupos = [
    mpatches.Patch(color=COLORES_GRUPO[g], label=g)
    for g in ORDEN_GRUPOS if g in COLORES_GRUPO
]
ax.legend(handles=leyenda_grupos + [
    mpatches.Patch(color='#90A4AE', label='Baseline'),
    mpatches.Patch(color='#CFD8DC', label='Zeros'),
], fontsize=8, ncol=4, loc='upper left')

# ── Panel 2: Mejora vs baseline (%) por grupo — boxplot
ax2 = axes[1]
grupos_presentes = [g for g in ORDEN_GRUPOS if g in df_etf['Grupo'].values]
datos_box = [df_etf[df_etf['Grupo'] == g]['Mejora_base_pct'].values for g in grupos_presentes]
bp = ax2.boxplot(datos_box, patch_artist=True, medianprops=dict(color='black', linewidth=1.8))
for patch, g in zip(bp['boxes'], grupos_presentes):
    patch.set_facecolor(COLORES_GRUPO.get(g, '#888'))
    patch.set_alpha(0.75)

ax2.axhline(0, color='red', linewidth=0.9, linestyle='--', label='Sin mejora (0 %)')
ax2.set_xticks(range(1, len(grupos_presentes) + 1))
ax2.set_xticklabels(grupos_presentes, fontsize=9)
ax2.set_ylabel('Mejora RMSE vs baseline (%)')
ax2.set_title('Distribución de la mejora vs baseline por grupo de activos', fontweight='bold', fontsize=11)
ax2.legend(fontsize=9)

# Anotar mediana
for i, d in enumerate(datos_box, 1):
    ax2.text(i, np.median(d) + 0.3, f'{np.median(d):.1f}%', ha='center', va='bottom', fontsize=8)

plt.suptitle(
    f'Análisis de resultados — Ridge 0 lags semanal  (alpha={ALPHA} | test {FECHA_TEST_INI} →)',
    fontsize=12, fontweight='bold'
)
plt.tight_layout()
plt.show()